# 02. 사진 한 장으로 터널 적응 부담(블랙홀·화이트홀) 재기

**터널 입구 사진**(밖에서 본 모습)은 가운데가 **검은 구멍**처럼 보인다 → 들어가는 순간 눈이 어둠에 적응해야 함 (**블랙홀 효과**)
**터널 출구 사진**(안에서 본 모습)은 가운데가 **하얀 구멍**처럼 보인다 → 나가는 순간 눈이 밝음에 적응해야 함 (**화이트홀 효과**)

이 노트북은 사진에서 **구멍(포털) 부분**과 **그 주변**의 밝기를 자동으로 찾아서
`밝기 비율 = 밝은 쪽 / 어두운 쪽`을 계산한다. 비율이 클수록 적응시 부담이 크다.

📚 근거: Mehri 외(2017)는 터널 입구 위험도를 `log(바깥 휘도 / 안쪽 휘도)`로 평가한다. 여기서 `log_ratio`가 같은 개념이다.

⚠️ **카메라 한계**: 실제 휘도 차이는 100배 이상 나기도 하지만, 사진은 0~255로 눌러 담고 하얗게 날아간 부분(255)은 더 밝아도 255로 찍힌다. 그래서 사진에서 잰 비율은 **실제보다 작게 나오는 최소값**이다. 같은 카메라 사진끼리 **비교**(터널 A vs B, 아침 vs 저녁)하는 용도로 쓴다.

사진 넣는 곳: `images/tunnel/` (jpg, png). 구글 드라이브에서 받은 사진도 여기에 넣으면 된다.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

IMG_DIR = Path("images/tunnel")
OUT_DIR = Path("data/video")
IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 구멍(포털) 자동 찾기

1. 사진을 흑백으로 바꾸고 살짝 흐리게 한다 (잡티 제거).
2. **Otsu 방법**으로 "어두운 부분 / 밝은 부분"을 가르는 기준 밝기를 자동으로 정한다.
3. 어두운 덩어리, 밝은 덩어리 중 **화면 가운데에 가깝고 적당히 큰 것**을 구멍 후보로 고른다.
4. 구멍 바로 바깥의 띠(링) 영역을 **주변**으로 보고 두 영역의 평균 밝기를 비교한다.
5. 어두운 구멍 → `entry`(블랙홀), 밝은 구멍 → `exit`(화이트홀).

자동으로 잘못 잡히면 `box=(x, y, 가로, 세로)`로 구멍 위치를 직접 지정할 수 있다.

In [ ]:
def _best_blob(mask, h, w):
    n, labels, stats, cents = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    best, best_score = None, 0
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA] / (h * w)
        if not 0.005 <= area <= 0.4:                      # 너무 작거나(잡티) 너무 크면(하늘 전체) 제외
            continue
        dx, dy = (cents[i, 0] - w / 2) / (w / 2), (cents[i, 1] - h / 2) / (h / 2)
        score = np.sqrt(area) * max(0, 1 - np.hypot(dx, dy))   # 가운데 가까울수록, 클수록 점수 ↑
        if score > best_score:
            best, best_score = labels == i, score
    return best


def measure_portal(gray, portal):
    ring = cv2.dilate(portal.astype(np.uint8), np.ones((41, 41), np.uint8)).astype(bool) & ~portal
    p, s = gray[portal].mean(), gray[ring].mean()
    return p, s, ring


def analyze_image(path, box=None, max_w=1280):
    img = cv2.imread(str(path))
    if img is None:
        raise IOError(f"사진을 열 수 없음: {path}")
    if img.shape[1] > max_w:
        img = cv2.resize(img, (max_w, int(img.shape[0] * max_w / img.shape[1])))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    if box is not None:
        x, y, bw, bh = box
        cands = [np.zeros_like(gray, bool)]
        cands[0][y: y + bh, x: x + bw] = True
    else:
        blur = cv2.GaussianBlur(gray, (0, 0), 3)
        thr, _ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        kernel = np.ones((7, 7), np.uint8)
        cands = []
        for m in (blur < thr, blur > thr):                  # 어두운 구멍, 밝은 구멍 둘 다 시도
            m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
            b = _best_blob(m, h, w)
            if b is not None:
                cands.append(b)
    if not cands:
        return {"file": Path(path).name, "type": "not_found"}, img, None, None

    # 후보 중 주변과 밝기 차이가 가장 큰 것을 구멍으로 확정
    results = [(portal, *measure_portal(gray, portal)) for portal in cands]
    portal, p, s, ring = max(results, key=lambda r: abs(np.log((r[1] + 1) / (r[2] + 1))))
    kind = "entry" if p < s else "exit"
    ratio = (max(p, s) + 1) / (min(p, s) + 1)
    row = {
        "file": Path(path).name,
        "type": kind,
        "portal_lum": round(float(p), 1),
        "surround_lum": round(float(s), 1),
        "lum_ratio": round(float(ratio), 2),
        "log_ratio": round(float(np.log10(ratio)), 3),
        "portal_area": round(float(portal.mean()), 4),
        "sat_ratio_all": round(float((gray >= 250).mean()), 4),   # 사진 전체 중 하얗게 날아간 비율
    }
    return row, img, portal, ring


def show(img, portal, ring, row):
    vis = img.copy()
    if portal is not None:
        vis[ring] = (0.6 * vis[ring] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)    # 주변 = 초록
        vis[portal] = (0.6 * vis[portal] + 0.4 * np.array([0, 0, 255])).astype(np.uint8)  # 구멍 = 빨강
    plt.figure(figsize=(7, 4))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{row['file']} | {row.get('type')} | 밝기비 {row.get('lum_ratio', '-')}배")
    plt.axis("off")
    plt.show()

## 2. 데모: 가짜 터널 입구·출구 사진으로 확인

빨간색 = 찾은 구멍, 초록색 = 비교한 주변.

In [ ]:
def make_demo_images():
    h, w = 400, 640
    # 입구: 밝은 바깥 + 가운데 검은 아치
    entry = np.full((h, w, 3), (200, 190, 180), np.uint8)
    entry[h // 2:] = (120, 120, 120)
    cv2.ellipse(entry, (w // 2, h // 2 + 60), (130, 150), 0, 180, 360, (25, 25, 25), -1)
    cv2.rectangle(entry, (w // 2 - 130, h // 2 + 60), (w // 2 + 130, h // 2 + 120), (25, 25, 25), -1)
    # 출구: 어두운 터널 벽 + 가운데 하얀 출구
    exit_ = np.full((h, w, 3), (45, 40, 35), np.uint8)
    cv2.ellipse(exit_, (w // 2, h // 2 + 30), (90, 100), 0, 180, 360, (255, 255, 255), -1)
    cv2.rectangle(exit_, (w // 2 - 90, h // 2 + 30), (w // 2 + 90, h // 2 + 70), (255, 255, 255), -1)
    exit_ = cv2.GaussianBlur(exit_, (0, 0), 4)
    for name, im in [("_demo_entry.png", entry), ("_demo_exit.png", exit_)]:
        noisy = cv2.add(im, np.random.randint(0, 10, im.shape, dtype=np.uint8))
        cv2.imwrite(str(IMG_DIR / name), noisy)

make_demo_images()
for name in ["_demo_entry.png", "_demo_exit.png"]:
    row, img, portal, ring = analyze_image(IMG_DIR / name)
    show(img, portal, ring, row)
    print(row)

## 3. 실제 사진 일괄 처리 → CSV

`images/tunnel/`의 사진을 전부 분석해서 `data/video/tunnel_image_check.csv`로 저장한다.
파일명에 정보를 넣어두면 나중에 비교가 쉽다. 예: `노선_터널명_방향_시간대_exit.jpg`

자동 인식이 틀린 사진은 `MANUAL_BOX`에 `{"파일명": (x, y, 가로, 세로)}`로 구멍 위치를 적어주면 된다.

In [ ]:
MANUAL_BOX = {}   # 예: {"gyeongbu_exit_0730.jpg": (500, 200, 280, 220)}

rows = []
files = sorted(p for p in IMG_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png") and not p.name.startswith("_"))
print(f"사진 {len(files)}장")
for p in files:
    try:
        row, img, portal, ring = analyze_image(p, box=MANUAL_BOX.get(p.name))
    except IOError as e:
        print("건너뜀:", e)
        continue
    rows.append(row)
    show(img, portal, ring, row)

if rows:
    check_df = pd.DataFrame(rows)
    check_df.to_csv(OUT_DIR / "tunnel_image_check.csv", index=False, encoding="utf-8-sig")
    display(check_df)
    display(check_df.groupby("type")[["lum_ratio", "log_ratio", "sat_ratio_all"]].describe().T)

## 읽는 법

- `lum_ratio` 2배 안팎: 적응 부담 작음 (흐린 날, 조명 좋은 터널)
- `lum_ratio` 5배 이상 + 출구 사진에서 `sat_ratio_all`이 큼: 나오자마자 하얗게 안 보이는 상황 → **사고 위험 구간 후보**
- 같은 터널을 **시간대별로** 찍은 사진이 있으면, 해가 출구 정면에 있는 시간에 비율이 튀는지 확인한다. 이게 우리 모델의 "태양 방위각 × 터널 출구 방향" 피처를 사진으로 확인하는 과정이다.